In [3]:
import cv2
import numpy as np

cap = cv2.VideoCapture("Transvers.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (800, 500))
    output = frame.copy()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 1. Strong blur (remove texture noise)
    blur = cv2.GaussianBlur(gray, (11,11), 0)

    # 2. Blackhat → highlight cracks
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17,17))
    blackhat = cv2.morphologyEx(blur, cv2.MORPH_BLACKHAT, kernel)

    # 3. Threshold
    _, thresh = cv2.threshold(blackhat, 30, 255, cv2.THRESH_BINARY)

    # 4. Morph close (connect cracks)
    kernel2 = np.ones((5,5), np.uint8)
    close = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel2, iterations=2)

    # 5. Morph open (remove small noise)
    clean = cv2.morphologyEx(close, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))

    # 6. Find contours
    contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Filter small + weird shapes
        if 300 < area < 8000:

            x,y,w,h = cv2.boundingRect(cnt)

            # shape filter (avoid square noise)
            if w > 15 and h > 15:

                #  Green crack outline
                cv2.drawContours(output, [cnt], -1, (0,255,0), 2)

                #  Red bounding box
                cv2.rectangle(output, (x,y), (x+w,y+h), (0,0,255), 2)

    cv2.imshow("Final Output", output)
    cv2.imshow("Mask", clean)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()